# **Flight Delay Dataset EDA**
#### *Numbers, correlations, and cutting the fat before modeling*

## **Setup**

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

pl.Config.set_tbl_rows(30)

## **1. Load + Sanity Check**

In [ ]:
df = pl.read_csv("../data/flightsData1m.csv")
print(df.shape)
df.head()

In [ ]:
df.schema

In [ ]:
null_counts = df.null_count().transpose(include_header=True, header_name="column", column_names=["null_count"])
null_counts = null_counts.with_columns((pl.col("null_count") / df.height * 100).alias("null_pct")).sort("null_pct", descending=True)
print(null_counts)

**Findings:** null pattern matches SQL exactly — `ArrDelay`/`ArrDelayMinutes`/`AirTime`/`ActualElapsedTime` all share the same 3.30% null rate (~32,974 rows), confirming these are all null together for cancelled/diverted flights, not a data quality issue.

## **2. Distribution Check**

### 2.1 DepDelay

In [ ]:
print(df["DepDelay"].describe())

plt.figure(figsize=(8,4))
plt.hist(df["DepDelay"].drop_nulls().to_numpy(), bins=50)
plt.title("DepDelay distribution")
plt.show()

**Findings:** mean 13.1 min, but median is **-2 min** — over half of all flights depart early or exactly on time. The mean is inflated by a long right tail of severe outliers (max 7223 min before the >720min outlier drop). This gap between mean and median is the clearest sign of the skew confirmed numerically in 2.3.

### 2.2 ArrDelay, TaxiOut, TaxiIn, Distance

In [ ]:
for col in ["ArrDelay", "TaxiOut", "TaxiIn", "Distance"]:
    print(f"--- {col} ---")
    print(df[col].describe())
    plt.figure(figsize=(8,4))
    plt.hist(df[col].drop_nulls().to_numpy(), bins=50)
    plt.title(f"{col} distribution")
    plt.show()

**Findings:**
- `ArrDelay`: same pattern as DepDelay — mean 7.6, median -5, most flights arrive early.
- `TaxiOut`: mean ~17 min, tighter spread (std 9.5) than delay columns — ground taxi time is far more consistent than delay.
- `TaxiIn`: mean ~7.9 min, similarly tight.
- `Distance`: mean ~798 mi, wide range (31 to 5095 mi) but much less extreme skew than the delay columns.

### 2.3 Skewness

In [ ]:
for col in ["DepDelay", "ArrDelay", "TaxiOut", "TaxiIn", "Distance"]:
    print(col, "skew:", df[col].skew())

**Findings:** `DepDelay` (13.0) and `ArrDelay` (11.8) are extremely right-skewed — far beyond typical "skewed" data. `TaxiOut`/`TaxiIn` are moderately skewed (3.7 / 5.0). `Distance` is only mildly skewed (1.5). Confirms delay columns need special handling (capping, log-transform, or bucketing into risk classes) before they'd work well in a linear model — tree-based models handle this skew natively, which is one more reason a RandomForest/LightGBM approach fits better here than linear regression.

### 2.4 Mean vs Median (outlier pull check)

In [ ]:
for col in ["DepDelay", "ArrDelay"]:
    print(col, "mean:", df[col].mean(), "median:", df[col].median())

**Findings:** confirms 2.1 — for both delay columns, mean is meaningfully higher than median, meaning a "typical" flight is on-time-or-early, and the average is being pulled upward by a relatively small number of severely delayed flights.

### 2.5 Categorical distributions

In [ ]:
print(df["Airline"].value_counts().sort("count", descending=True))
print(df["Cancelled"].value_counts())
print(df["Diverted"].value_counts())

**Findings:** Airline volume ranges from Southwest (~180K) down to GoJet (~8K) — matches SQL Category 1 findings, no surprises. `Cancelled` and `Diverted` are both heavily imbalanced (small minority `true`) — expected, and important to remember if either becomes a classification target later: will need class weighting or resampling.

## **3. Outlier Handling (from SQL decision)**

In [ ]:
# Dropping DepDelay > 720 minutes (12 hrs) — matches the SQL-layer decision
before = df.height
df = df.filter((pl.col("DepDelay").is_null()) | (pl.col("DepDelay") <= 720))
after = df.height
print(f"Dropped {before - after} rows ({(before-after)/before*100:.3f}%)")

**Result:** dropped 1,026 rows (0.103%), matching the SQL-layer count exactly. Consistent between the two layers, as expected.

## **4. Feature-Feature Correlation (find redundant columns)**

In [ ]:
numeric_cols = [c for c, dt in df.schema.items() if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32)]
print(numeric_cols)

In [ ]:
corr_df = df.select(numeric_cols).drop_nulls()
corr_matrix = np.corrcoef(corr_df.to_numpy(), rowvar=False)
corr_pl = pl.DataFrame(corr_matrix, schema=numeric_cols)
corr_pl

In [ ]:
# Flag pairs with |correlation| > 0.9 (excluding self-correlation)
threshold = 0.9
redundant_pairs = []
for i, col_i in enumerate(numeric_cols):
    for j, col_j in enumerate(numeric_cols):
        if i < j and abs(corr_matrix[i, j]) > threshold:
            redundant_pairs.append((col_i, col_j, round(corr_matrix[i, j], 3)))

for pair in redundant_pairs:
    print(pair)

**Findings:** several near-duplicate columns confirmed —
- `DepDelayMinutes` ↔ `DepDelay` (0.997) — same value, DepDelayMinutes just clips negatives to 0
- `DepDelay` ↔ `ArrDelay` (~0.95), `DepDelayMinutes` ↔ `ArrDelayMinutes` (0.968) — departure delay strongly predicts arrival delay, as expected
- `CRSDepTime` ↔ `DepTime` ↔ `WheelsOff` (0.92–0.97) — scheduled time, actual time, and taxi-out start time cluster together, since they're all clock-time-of-day values

**This matters for Section 7:** these correlated pairs are mostly *outcome* columns (actual delay/time data), not features you'd have *before* a flight happens — flagged further below.

## **5. Feature-Target Correlation**

**Target:** `ArrDelay` (continuous). Before reading the ranking below — a critical catch first.

In [ ]:
target = "ArrDelay"
target_corr = {}
for col in numeric_cols:
    if col == target:
        continue
    sub = df.select([col, target]).drop_nulls()
    if sub.height > 0:
        c = np.corrcoef(sub[col].to_numpy(), sub[target].to_numpy())[0, 1]
        target_corr[col] = round(c, 3)

target_corr_sorted = dict(sorted(target_corr.items(), key=lambda x: abs(x[1]), reverse=True))
for k, v in target_corr_sorted.items():
    print(k, v)

**⚠️ Data leakage warning:** the top-correlated columns (`ArrDelayMinutes` 0.979, `DepDelay` 0.95, `DepDelayMinutes` 0.948, `ArrivalDelayGroups` 0.942, `DepartureDelayGroups` 0.89, `ArrDel15` 0.621) are **not real predictive features** — they're either:
1. The same value as the target stored differently (`ArrDelayMinutes` is `ArrDelay` clipped at 0), or
2. Only known *after* the flight has already been delayed/departed (`DepDelay`, `TaxiOut`, `WheelsOff`, `ArrTime`, etc.)

The project's actual goal — a control tower that predicts risk **before** a flight, from route + cargo inputs — needs features known ahead of time only. Everything with real signal in this ranking is either leakage or a downstream outcome. **Section 7 below is corrected to exclude these.**

## **6. Categorical Feature Relevance**

In [ ]:
categorical_check_cols = ["Airline", "Origin", "Dest", "DepTimeBlk", "DayOfWeek"]

for col in categorical_check_cols:
    print(f"--- avg {target} by {col} (top 10) ---")
    result = (
        df
        .filter(pl.col(target).is_not_null())
        .group_by(col)
        .agg([pl.len().alias("count"), pl.col(target).mean().alias("avg_target")])
        .filter(pl.col("count") > 200)
        .sort("avg_target", descending=True)
        .head(10)
    )
    print(result)

**Findings:** Airline ranking (Allegiant, JetBlue, Frontier worst) matches SQL Category 1 exactly — good cross-validation between the two layers. These categorical groupings are genuinely useful, unlike most of Section 5's numeric ranking — `Airline`, `Origin`, `Dest`, and time-of-day (`DepTimeBlk`) are all known before a flight and all show real variation in average delay, making them strong candidate features.

## **7. Baseline Model + Feature Importance (corrected — no leakage)**

Restricting to features known **before departure** — this excludes every actual/outcome column (`DepDelay`, `TaxiOut`, `TaxiIn`, `AirTime`, `WheelsOff`, `WheelsOn`, `ActualElapsedTime`, `ArrTime`, `DepTime`, and every `*Minutes`/`*Del15`/`*DelayGroups` variant), since a real control-tower prediction wouldn't have access to any of them yet.

In [ ]:
pre_flight_numeric = [
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance", "DistanceGroup",
    "Month", "Quarter", "DayOfWeek", "DayofMonth", "Year",
]

pre_flight_categorical = ["Airline", "Origin", "Dest"]

print("Pre-flight numeric features:", pre_flight_numeric)
print("Pre-flight categorical features:", pre_flight_categorical)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

model_df = df.select(pre_flight_numeric + pre_flight_categorical + [target]).drop_nulls()

encoders = {}
X_df = model_df.select(pre_flight_numeric).clone()
for col in pre_flight_categorical:
    le = LabelEncoder()
    encoded = le.fit_transform(model_df[col].to_numpy())
    X_df = X_df.with_columns(pl.Series(col, encoded))
    encoders[col] = le

feature_cols = pre_flight_numeric + pre_flight_categorical
X = X_df.select(feature_cols).to_numpy()
y = model_df[target].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("Train R^2:", rf.score(X_train, y_train))
print("Test R^2:", rf.score(X_test, y_test))

**Expect a much lower, more realistic R² here** (likely in the 0.05-0.20 range, not 0.99) — that's not a worse result, it's the *honest* one. Flight delay is genuinely hard to predict from schedule-only information; the earlier 0.994 was an artifact of leakage, not real model skill. A modest R² here is expected and defensible — document it as such in your report rather than chasing an artificially high score.

In [ ]:
importances = sorted(zip(feature_cols, rf.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in importances:
    print(f"{feat}: {imp:.4f}")

**Findings:** *(run and fill in — expect `Origin`/`Dest`/`Airline` and `CRSDepTime` to rank highest, consistent with the SQL findings that airport, carrier, and hour-of-day were the strongest patterns found)*

## **8. Finalize Feature Set**

In [ ]:
final_features = [
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "Month", "DayOfWeek", "Airline", "Origin", "Dest",
]

dropped_features = {
    "DepDelayMinutes": "duplicate of DepDelay (0.997 corr), same info clipped differently",
    "ArrDelayMinutes": "duplicate of target ArrDelay, same info clipped differently — leakage",
    "DepDelay": "outcome column, not known before departure — leakage for a pre-flight prediction",
    "TaxiOut / TaxiIn / AirTime / ActualElapsedTime": "outcome columns, only known during/after the flight",
    "WheelsOff / WheelsOn / ArrTime / DepTime (actual)": "actual timestamps, only known during/after the flight",
    "ArrDel15 / DepDel15 / ArrivalDelayGroups / DepartureDelayGroups": "derived directly from the delay outcome — leakage",
    "DistanceGroup": "redundant with Distance (bucketed version of the same value)",
    "Quarter / DayofMonth / Year": "redundant with Month/DayOfWeek or near-constant (single year in this sample)",
    "OriginAirportID / DestAirportID / *SeqID / *CityMarketID / *Wac / *StateFips": "ID-encoded duplicates of Origin/Dest text columns",
    "DOT_ID_* / Flight_Number_* / Tail_Number": "administrative identifiers, no delay signal expected",
}

print("Final features:", final_features)
print("Dropped:", list(dropped_features.keys()))

## **9. Summary**

**Key findings:**
- Delay data (`DepDelay`, `ArrDelay`) is extremely right-skewed (skew 11-13) — median is negative, mean is pulled up by a long tail of severe outliers. Confirmed the SQL-layer decision to drop the 1,026 rows (0.103%) with DepDelay > 720 min.
- Feature-feature correlation revealed several redundant column pairs, mostly variants of the same underlying delay/time value stored differently.
- **Critical catch:** the naive feature-target correlation and first baseline model (R² = 0.994) were built on leaked/outcome columns — `ArrDelayMinutes` is essentially the target itself renamed, and most other top-ranked columns (`DepDelay`, `TaxiOut`, actual timestamps) are only known *after* a flight has already begun, making them useless for a pre-flight control-tower prediction.
- Corrected the feature set to pre-flight-only information (schedule, route, carrier, calendar) and retrained — a lower, more honest R² is expected and should be reported as such, not treated as a failure.
- Categorical findings (Airline, Origin/Dest rankings) cross-validate cleanly against the SQL EDA report — same airlines and airports show up as worst performers in both layers.

**Target:** `ArrDelay` (continuous) — confirm this is still the intended target, or revisit as a risk-classification bucket (Low/Med/High) if that fits the control tower app's UI better.

**Final feature set:** `CRSDepTime`, `CRSArrTime`, `CRSElapsedTime`, `Distance`, `Month`, `DayOfWeek`, `Airline`, `Origin`, `Dest` — 9 features, all knowable before a flight departs.